## Importing needed libraries and the dataset from the ECDC.



In [50]:
import pandas as pd
import numpy as np
from scipy import stats


In [35]:
# importing dataset

res_data = pd.read_csv("../data/raw/ecdc_surveillance_data_Antimicrobial_resistance.csv")

## Exploratory Data Analysis On Dataset

In [36]:
res_data.head()

,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
0,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,DE,Germany,0.000000000,NaN
1,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,EE,Estonia,0.000000000,NaN
2,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,EL,Greece,7.000000000,NaN
3,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,NL,Netherlands,1.000000000,NaN
4,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"I - 'susceptible, increased exposure' isolates",N,2000,NO,Norway,24.000000000,NaN


In [37]:
res_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3635 entries, 0 to 3634
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   HealthTopic  3635 non-null   str    
 1   Population   3635 non-null   str    
 2   Indicator    3635 non-null   str    
 3   Unit         3635 non-null   str    
 4   Time         3635 non-null   int64  
 5   RegionCode   3635 non-null   str    
 6   RegionName   3635 non-null   str    
 7   NumValue     3635 non-null   str    
 8   TxtValue     0 non-null      float64
dtypes: float64(1), int64(1), str(7)
memory usage: 255.7 KB


In [38]:
res_data.duplicated().any()

np.False_

In [39]:
# Converting NumValue variable from string to float

res_data['NumValue'] = pd.to_numeric(res_data['NumValue'], errors='coerce')

In [40]:
res_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3635 entries, 0 to 3634
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   HealthTopic  3635 non-null   str    
 1   Population   3635 non-null   str    
 2   Indicator    3635 non-null   str    
 3   Unit         3635 non-null   str    
 4   Time         3635 non-null   int64  
 5   RegionCode   3635 non-null   str    
 6   RegionName   3635 non-null   str    
 7   NumValue     3627 non-null   float64
 8   TxtValue     0 non-null      float64
dtypes: float64(2), int64(1), str(6)
memory usage: 255.7 KB


In [41]:
# Investigating which values in the NumValue are null
# There are all in the indicaator 'R - resistant isolates, percentage'. This indicator value isn't needed in our project analysis

res_data[res_data['NumValue'].isna()]

,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
1455,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2000,EE,Estonia,NaN,NaN
1459,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2000,SE,Sweden,NaN,NaN
1511,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2003,CY,Cyprus,NaN,NaN
1702,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2009,SE,Sweden,NaN,NaN
2079,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2022,LI,Liechtenstein,NaN,NaN
2108,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2023,LI,Liechtenstein,NaN,NaN
2138,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2024,LI,Liechtenstein,NaN,NaN
2168,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2025,LI,Liechtenstein,NaN,NaN


### Extracting Needed Indicators For This Project / Calculating Confidence Interval
 The variables needed in other to estimate the proportion of E. coli bacterial isolates that are resistant to Third-generation cephalosporins
 - Resistant Isolates 
 - Total Isolates
 - For a period of 2020 -2025


In [51]:
# Extracting needed variables

study_years = [2020, 2021, 2022, 2023,2024, 2025]

year = res_data[res_data['Time'].isin(study_years)]

needed_indicators= ['R - resistant isolates', 'Total tested isolates']
analysis_df = year[year['Indicator'].isin(needed_indicators)]

analysis_df.sample(10)



,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
1279,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2020,BG,Bulgaria,108.0,NaN
1342,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2022,EE,Estonia,118.0,NaN
3492,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2021,DE,Germany,29021.0,NaN
1327,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2021,NL,Netherlands,437.0,NaN
3550,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2023,CZ,Czechia,3892.0,NaN
1403,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2024,ES,Spain,1755.0,NaN
3505,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2021,LU,Luxembourg,354.0,NaN
3496,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2021,ES,Spain,7425.0,NaN
3633,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,Total tested isolates,N,2025,SI,Slovenia,1643.0,NaN
1371,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,R - resistant isolates,N,2023,DK,Denmark,333.0,NaN


In [43]:
# Confirming that both the resistant and total testes isolates have the same count values per year

analysis_gb = analysis_df.groupby(['Time', 'Indicator'])['NumValue'].count().reset_index()

analysis_gb

,Time,Indicator,NumValue
0,2020,R - resistant isolates,29
1,2020,Total tested isolates,29
2,2021,R - resistant isolates,29
3,2021,Total tested isolates,29
4,2022,R - resistant isolates,30
5,2022,Total tested isolates,30
6,2023,R - resistant isolates,29
7,2023,Total tested isolates,29
8,2024,R - resistant isolates,30
9,2024,Total tested isolates,30


In [44]:
# Extracting my point estimate

resistant = analysis_df[analysis_df['Indicator']== 'R - resistant isolates']

resistant_S = resistant['NumValue'].sum()

total = analysis_df[analysis_df['Indicator']== 'Total tested isolates']

total_S = total['NumValue'].sum()

print(resistant_S)
print(total_S)

136656.0
986328.0


In [45]:
p = resistant_S / total_S


print(p)


0.13855025914300315


In [46]:
# Chosen confidence level is 0.95 (95%). zscore fot this confidence level is 1.96
# Calculating Standard Error

zscore = 1.96

se = np.sqrt((p * (1-p))/total_S)

se

np.float64(0.00034786277930708945)

In [47]:
# Calculating margin of Error

me = zscore * se

me

np.float64(0.0006818110474418953)

In [48]:
# Calculating the confidence interval

lower_confidence, higher_confidence= stats.norm.interval(0.95, p, se)

print(lower_confidence)
print(higher_confidence)

0.13786846062399924
0.13923205766200705


In [49]:
analysis_df.to_csv('analysis_df.csv', index= False)